# Paper Figure Console

This notebook is the paper-side console for the repository. It loads saved artifacts from `results/`, calls plotting utilities from `src.plotting`, previews the figures, and exports them for the paper.

## Environment Check

Run this cell first to locate the project root, expose the source package, and confirm the local notebook environment.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import plotly
from IPython.display import HTML, display


def find_project_root(start_path: Path) -> Path:
    for candidate in [start_path, *start_path.parents]:
        if (candidate / "src" / "model").exists() and (candidate / "experiment.ipynb").exists():
            return candidate
    raise RuntimeError("Could not locate the project root from the current working directory.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.plotting import generate_default_paper_figures, load_paper_figure_data, save_figures

pd.set_option("display.max_colwidth", 160)

versions_df = pd.DataFrame(
    [
        {"package": "python", "version": sys.version.split()[0]},
        {"package": "pandas", "version": pd.__version__},
        {"package": "plotly", "version": plotly.__version__},
    ]
)
display(versions_df)
print(f"Project root: {PROJECT_ROOT}")
print(f"Python executable: {sys.executable}")


## Configuration

Set the experiment name, the seeds to load, the seed used for qualitative plots, and the case index to visualize.

In [ ]:
EXPERIMENT_NAME = "distilbert_sst2_multiseed"
SEEDS = [42, 43, 44, 45]
INSPECT_SEED = 42
QUALITATIVE_CASE_INDEX = 519
TOP_N_TOKENS = 8
FIGURE_OUTPUT_DIR = PROJECT_ROOT / "paper" / "figures"

print(f"Experiment name: {EXPERIMENT_NAME}")
print(f"Seeds: {SEEDS}")
print(f"Inspect seed: {INSPECT_SEED}")
print(f"Qualitative case index: {QUALITATIVE_CASE_INDEX}")
print(f"Figure output directory: {FIGURE_OUTPUT_DIR}")


## Load Artifact Tables

This cell loads validation metrics, ranking-similarity summaries, deletion summaries, and the seed-specific files needed for qualitative plots.

In [ ]:
data = load_paper_figure_data(
    project_root=PROJECT_ROOT,
    experiment_name=EXPERIMENT_NAME,
    seeds=SEEDS,
    inspect_seed=INSPECT_SEED,
)

display(data.validation_metrics)
display(data.validation_metrics_aggregate)
display(data.ranking_similarity_aggregate)
display(
    data.deletion_aggregate[
        data.deletion_aggregate["k_setting_type"].eq("top_k")
        & data.deletion_aggregate["actual_k"].isin([1, 2, 3])
    ].sort_values(["method", "actual_k"]).reset_index(drop=True)
)


## Generate And Preview Figures

The notebook below behaves like a console: it asks `src.plotting` for the default figure bundle and previews every figure inline.

In [ ]:
def display_plotly_figure(fig):
    try:
        fig.show()
    except ValueError as exc:
        if "nbformat" not in str(exc):
            raise
        display(HTML(fig.to_html(include_plotlyjs="cdn")))


figures = generate_default_paper_figures(
    data,
    qualitative_case_index=QUALITATIVE_CASE_INDEX,
    top_n_tokens=TOP_N_TOKENS,
)

for figure_name, figure in figures.items():
    print(f"=== {figure_name} ===")
    display_plotly_figure(figure)


## Export Figures

Run this cell to save every figure to `paper/figures/` as HTML, and as PNG when the local image export backend is available.

In [ ]:
manifest_df = save_figures(figures, FIGURE_OUTPUT_DIR, save_png=True, save_html=True)
display(manifest_df)
